In [8]:
%load_ext autoreload
%autoreload 2

In [4]:
import torch
import torch.nn as nn

class ScamGuardLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(ScamGuardLSTM, self).__init__()
        
        # 1. Embedding Layer: Converts token integers into dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. LSTM Layer: The sequential "memory" of the model
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        # 3. Fully Connected Layer: Maps the LSTM output to our classes (0 = Safe, 1 = Scam)
        # Multiply hidden_dim by 2 because it's bidirectional
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
        # 4. Dropout: Regularization to prevent overfitting on small datasets
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text, text_lengths):
        # text shape: [batch_size, sequence_length]
        embedded = self.dropout(self.embedding(text))
        
        # Pack sequence to handle variable message lengths efficiently in memory
        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded, text_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        packed_output, (hidden, cell) = self.lstm(packed_embedded)
        
        # Extract the final hidden state from both the forward and backward passes
        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
            
        # Pass through the linear layer for the final logit prediction
        return self.fc(hidden)


In [2]:
import sys
import os

data_path = os.path.abspath(os.path.join('..', 'data'))

if data_path not in sys.path:
    sys.path.append(data_path)

from data_loader import load_scam_data, load_embeddings

dataset = load_scam_data()
glove = load_embeddings()

glove_weights = glove.vectors

ModuleNotFoundError: No module named 'data_loader'

In [6]:
import torch.optim as optim
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ScamGuardLSTM(
    vocab_size=len(glove.stoi), 
    embedding_dim=100, 
    hidden_dim=256, 
    output_dim=1, 
    n_layers=2, 
    bidirectional=True, 
    dropout=0.5
)
model.embedding.weight.data.copy_(glove_weights) # Load pre-trained GloVe
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss() # Best for binary classification (Scam vs. Safe)

def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0
    
    for batch in iterator:
        optimizer.zero_grad() # Reset gradients from last step
        
        # 'text' are the word indices, 'lengths' are message lengths
        predictions = model(batch.text, batch.lengths).squeeze(1)
        
        loss = criterion(predictions, batch.label.float())
        loss.backward() # Backpropagation
        optimizer.step() # Update model weights
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

NameError: name 'glove' is not defined